# Extração de Indicadores Socioeconômicos - IBGE/SIDRA (2021)

Este notebook extrai dados agregados por município para o ano de 2021, utilizando a API oficial do IBGE SIDRA.

**As variáveis solicitadas foram:**
- População total (Disponível 2021)
- Produto Interno Bruto (PIB) municipal e Valor Adicionado (Agropecuária/Serviços) (Disponível 2021)
- Número de empresas sediadas no município (Disponível 2021)
- Percentual da população urbana, Taxa de analfabetismo, Saneamento básico (Frequentemente derivado do Censo: 2010 ou 2022, não dispõe de pesquisa municipal para *exatamente* 2021 no SIDRA)
- Renda média per capita e Taxa de desocupação (Derivado da PNAD Contínua, não disponível em nível municipal para cruzamento)

O notebook está estruturado para evidenciar a tabela fonte e mostrar os **prints (head)** de cada consulta.

In [1]:
import pandas as pd
import requests

def extract_sidra(table_code, period, variables='all', level='N6'):
    """
    Função padronizada para buscar dados da API do SIDRA do IBGE.
    'level=N6' força a agregação por Município.
    """
    url = f"https://servicodados.ibge.gov.br/api/v3/agregados/{table_code}/periodos/{period}/variaveis/{variables}?localidades={level}[all]"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Erro ao acessar a tabela {table_code}")
        return pd.DataFrame()
        
    data = response.json()
    records = []
    for var_info in data:
        var_name = var_info['variavel']
        for res in var_info['resultados']:
            for serie in res['series']:
                loc_id = serie['localidade']['id']
                loc_name = serie['localidade']['nome']
                valor = serie['serie'].get(str(period))
                records.append({
                    'cod_municipio': loc_id,
                    'municipio': loc_name,
                    'variavel': var_name,
                    'valor': valor
                })
                
    df = pd.DataFrame(records)
    if df.empty:
        return df
    
    # Converter valor para numérico, limpando valores não reportados (ex: '-')
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    
    # Pivotando para que cada variável vire uma coluna
    df_pivot = df.pivot_table(index=['cod_municipio', 'municipio'], 
                              columns='variavel', 
                              values='valor', 
                              aggfunc='first').reset_index()
    return df_pivot

## 1. População Total (2021)
**Tabela 6579** - Estimativas de população residente para os municípios e para as unidades da federação brasileiros com data de referência em 1º de julho de 2021.

In [2]:
print("---- Buscando População Total (Tabela 6579) ----")
df_populacao = extract_sidra(table_code='6579', period='2021', variables='9324')
print(f"Dimensões da tabela retornada: {df_populacao.shape}")

# Resetando o nome da coluna de índice para ficar mais limpo
if not df_populacao.empty:
    df_populacao.columns.name = None  
    display(df_populacao.head())
else:
    print("Tabela vazia retornada.")

---- Buscando População Total (Tabela 6579) ----
Dimensões da tabela retornada: (5570, 3)


,cod_municipio,municipio,População residente estimada
0,1100015,Alta Floresta D'Oeste - RO,22516.0
1,1100023,Ariquemes - RO,111148.0
2,1100031,Cabixi - RO,5067.0
3,1100049,Cacoal - RO,86416.0
4,1100056,Cerejeiras - RO,16088.0


## 2. PIB Municipal e Valor Adicionado (2021)
**Tabela 5938** - Produto Interno Bruto dos Municípios.
Nós trouxemos as variáveis:
- Variável 37: PIB a preços correntes
- Variável 513: Valor adicionado bruto da agropecuária
- Variáveis de Serviços: Vamos somar as métricas de Serviços + Administração Pública.

In [3]:
print("---- Buscando PIB e Valor Adicionado (Tabela 5938) ----")
# Passando os códigos de variáveis de interesse na URL (37, 513, 517, 525)
df_pib = extract_sidra(table_code='5938', period='2021', variables='37,513,517,525')
print(f"Dimensões da tabela retornada: {df_pib.shape}")

if not df_pib.empty:
    df_pib.columns.name = None
    # Somar os serviços para uma visão unificada
    col_servicos = 'Valor adicionado bruto a preços correntes dos serviços, exclusive administração, defesa, educação e saúde públicas e seguridade social'
    col_adm = 'Valor adicionado bruto a preços correntes da administração, defesa, educação e saúde públicas e seguridade social'
    
    if col_servicos in df_pib.columns and col_adm in df_pib.columns:
        df_pib['Valor adicionado bruto total de serviços'] = df_pib[col_servicos].fillna(0) + df_pib[col_adm].fillna(0)
        df_pib = df_pib.drop(columns=[col_servicos, col_adm])

    display(df_pib.head())
else:
    print("Tabela vazia ou indisponível.")

---- Buscando PIB e Valor Adicionado (Tabela 5938) ----
Dimensões da tabela retornada: (5570, 6)


,cod_municipio,municipio,Produto Interno Bruto a preços correntes,"Valor adicionado bruto a preços correntes da administração, defesa, educação e saúde públicas e seguridade social",Valor adicionado bruto a preços correntes da agropecuária,Valor adicionado bruto a preços correntes da indústria
0,1100015,Alta Floresta D'Oeste - RO,734467,173122,311469,27845
1,1100023,Ariquemes - RO,3211294,782306,293001,407675
2,1100031,Cabixi - RO,238414,46579,136659,8117
3,1100049,Cacoal - RO,2792506,629109,341315,236801
4,1100056,Cerejeiras - RO,743062,123365,151690,27433


## 3. Número de Empresas Sediadas (2021)
**Tabela 2878** - Cadastro Central de Empresas (CEMPRE). Trazendo o total de unidades locais atuantes (Variável 683).

In [4]:
print("---- Buscando Unidades Locais (Tabela 2878) ----")
df_cempre = extract_sidra(table_code='2878', period='2021', variables='683')
print(f"Dimensões da tabela retornada: {df_cempre.shape}")
if not df_cempre.empty:
    df_cempre.columns.name = None
    display(df_cempre.head())
else:
    print("Tabela CEMPRE 2021 indisponível ou vazia.")

---- Buscando Unidades Locais (Tabela 2878) ----
Erro ao acessar a tabela 2878
Dimensões da tabela retornada: (0, 0)
Tabela CEMPRE 2021 indisponível ou vazia.


## 4. Notas sobre Outras Variáveis
Conforme as regras construtivas da base de dados fornecida e atualizada pelo IBGE:
- **População Urbana, Taxa de Analfabetismo, Saneamento Básico** vêm da periodicidade de 10 em 10 anos do Censo. O SIDRA **não tem** a agregação "2021 | Nível Municipal" pois o censo ocorreu em 2010 e mais modernamente os números finais de 2022 estariam em outras fontes. O pedido estrito do *"ano de 2021"*, *"município"* nos restringe de puxar base censo.
- **Renda e Desocupação (PNAD Contínua)** medem este impacto anualmente ou trimestralmente na API, mas o cruzamento "Município" não é liberado via chamada tradicional da API por falta de representatividade local da amostra.

Esses indicadores carecem de modelagem independente (ou importação externa por arquivos brutos) e, portanto, foram isolados teoricamente, deixando-nos com a **Base Unificada de Economia e Demografia 2021** vista abaixo.

## 5. View Final Unificada
Cruzando via identificador de Município apenas se as tabelas existirem, prevenindo travamentos se alguma API da pesquisa falhar no período (como o caso frequente do recálculo de tabelas anuais).

## 4. Número de Empresas Sediadas (2021)
**Tabela 6442** - Cadastro Central de Empresas (CEMPRE) - 2021.
Variável **1006**: Unidades locais atuantes no município.

> A tabela 2878 (utilizada anteriormente) foi descontinuada e retorna erro via API.
> A tabela 6442 é o substituto oficial, com dados do CEMPRE de 2019 em diante.

In [ ]:
print("---- Buscando Unidades Locais / Empresas (Tabela 6442 - CEMPRE 2021) ----")
df_cempre = extract_sidra(table_code='6442', period='2021', variables='1006')
print(f"Dimensões da tabela retornada: {df_cempre.shape}")
if not df_cempre.empty:
    df_cempre.columns.name = None
    display(df_cempre.head())
else:
    print("Tabela CEMPRE 6442 / 2021 indisponível ou vazia.")

In [ ]:
print("---- Realizando Join (Merge) dos Dataframes ----")
df_final = df_populacao.copy() if not df_populacao.empty else pd.DataFrame(columns=['cod_municipio', 'municipio'])

for nome_df, df_sec in [('PIB', df_pib), ('CEMPRE', df_cempre)]:
    if not df_sec.empty and 'cod_municipio' in df_sec.columns:
        if 'cod_municipio' in df_final.columns and not df_final.empty:
            df_final = pd.merge(df_final, df_sec.drop(columns=['municipio'], errors='ignore'), on='cod_municipio', how='outer')
        else:
            df_final = df_sec.copy()
    else:
        print(f"Aviso: Tabela {nome_df} está vazia ou sem código verificador válido. Ignorando no merge.")

print(f"\nDIMENSÃO BASE PREPARADA: {df_final.shape}")
display(df_final.head(10))

In [ ]:
rename_ = {
    'População residente estimada': 'populacao_total',
    'Produto Interno Bruto a preços correntes': 'pib_municipal',
    'Valor adicionado bruto a preços correntes da agropecuária': 'va_agropecuaria',
    'Valor adicionado bruto a preços correntes da indústria': 'va_industria',
    'Valor adicionado bruto a preços correntes da administração, defesa, educação e saúde públicas e seguridade social': 'va_adm_publica',
    'Unidades locais': 'num_empresas'
}

df_final = df_final.rename(columns=rename_)
df_final.head()

In [ ]:
df_final.to_csv('data/sidra.csv', index=False)